# M8_L2 - scRNA-Seq Data Pre-Processing and QC

In M8_L1 we saw how a 10x Genomics library is built, and that each read pair carries a cell barcode, a UMI and a piece of cDNA. In this lab we work with a real *Arabidopsis* root dataset: we look at how the study was designed, download part of one sample, check the reads, and turn them into a genes × cells count matrix with **STARsolo**.

**NOTE:** This notebook uses the Bash kernel. The tools are in the `module_8_scRNA` conda environment (`environment/module_8_scrna.yml`). Create the Jupyter session with **8 CPU cores and at least 32 GB of RAM**, and allow 1 to 2 hours, most of which is waiting for downloads and STAR.

<u>**By the end of this notebook, you should have completed these tasks:**</u>
- Describe the design of the Ryu *et al.* (2019) root scRNA-seq study (GSE123013) and find its runs in SRA
- Download a 10-million-read subset of one sample as 10x-style R1, R2 and I1 FASTQ files
- Check the read lengths and quality, and explain which FastQC warnings are expected for 10x data
- Explain why 10x reads are not trimmed in the same way as bulk RNA-Seq reads
- Build a STAR index for TAIR10 and run STARsolo with the 10x v2 barcode whitelist
- Read the STARsolo summary and compare your cells with the authors' Cell Ranger results

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

## 1. The dataset: Ryu *et al.* (2019), GSE123013

**Ryu KH, Huang L, Kang HM, Schiefelbein J (2019).** Single-cell RNA sequencing resolves molecular relationships among individual plant cells. *Plant Physiology* 179:1444-1456. [doi:10.1104/pp.18.01482](https://doi.org/10.1104/pp.18.01482) (open access: [PMC6446759](https://pmc.ncbi.nlm.nih.gov/articles/PMC6446759/))

This was one of the first studies to apply a commercial droplet platform (10x Genomics Chromium) to plants. The authors asked whether scRNA-seq could:
1. Recover all the major cell types of the *Arabidopsis* root from protoplasts
2. Find rare cell types, such as the quiescent center
3. Order root epidermal cells along their development, from the meristem to mature root-hair (trichoblast) and non-hair (atrichoblast) cells
4. Show, cell type by cell type, what happens in mutants that change epidermal cell fate

**Design.** Root tips of 5-day-old seedlings were digested to protoplasts and loaded on a Chromium controller with the **Single Cell 3' v2** kit. The libraries were sequenced on an **Illumina HiSeq 4000**, and reads were processed with **Cell Ranger 2.1.1** against TAIR10. The wild-type plants carry a *WER::GFP* reporter (the *WEREWOLF* promoter driving GFP in non-hair cells) in the Col-0 background. The two mutants affect the epidermis:
- ***rhd6-1*** (ROOT HAIR DEFECTIVE 6): makes almost no root hairs
- ***gl2*** (GLABRA2): non-hair cells are converted into hair-forming cells

| GEO sample | Sample | Genotype | SRA run(s) | Original FASTQ prefix | Cells (Cell Ranger, GEO matrix) |
|---|---|---|---|---|---|
| [GSM3490689](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3490689) | WT_rep1 | Col-0 *WER::GFP* | SRR8257100, SRR8257101 (two lanes) | `WT-WERGFP` | 4,406 |
| [GSM3490690](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3490690) | WT_rep2 | Col-0 *WER::GFP* | **SRR8257102** | `WT-WERGFP_2` | 1,473 |
| [GSM3490691](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3490691) | WT_rep3 | Col-0 *WER::GFP* | SRR8257103 | `WT-WERGFP_3` | 1,643 |
| [GSM3490692](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3490692) | rhd6-1 | *rhd6-1* | SRR8257104, SRR8257105 (two lanes) | `rhd6` | 2,505 |
| [GSM3490693](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3490693) | gl2 | *gl2* | SRR8257106 | `gl2` | 1,003 |

The three wild-type replicates add up to the **7,522 cells** reported in the paper. In this lab we use **WT_rep2 (SRR8257102)**, the smallest wild-type run (105 million read pairs). In M8_P1 we will analyze the authors' count matrices for all three wild-type replicates.

## 2. Set up the project folder and environment

We use the same project layout as in Modules 3 and 5. If you have not yet created the environment, do so once from the repository root:

```bash
module load mamba
mamba env create -f environment/module_8_scrna.yml
```

In [ ]:
# load mamba and activate the Module 8 environment
ml mamba
source activate module_8_scRNA
export TERM=xterm

# repository data/ folder (where fetch_data.sh saves files) and the project folder
REPO_DIR=${REPO_DIR:-$(cd ../../.. && pwd)}
DATA_DIR=${DATA_DIR:-$REPO_DIR/data}
PROJ=${WORK_DIR:-$HOME}/m8_scrna_project
mkdir -p $PROJ/{data/raw,references,results/{qc,starsolo},scripts}
cd $PROJ
pwd

## 3. Finding the runs in SRA

As in M3L2, we can ask NCBI for the run table of the study with Entrez Direct. Note the `spots` and `bases` columns, and that each run is listed as `PAIRED` even though the "pair" is really a barcode read plus a cDNA read.

In [ ]:
esearch -db sra -query SRP171040 | efetch -format runinfo > references/SraRunInfo_GSE123013.csv
# columns: Run, spots, bases, avgLength, LibraryLayout, Model, SampleName (GSM)
cut -d, -f1,4,5,7,16,20,30 references/SraRunInfo_GSE123013.csv | column -s, -t

```{note}
**Where are the barcodes?** If you download these runs from ENA, you get a single FASTQ file per run that contains only the cDNA read. The cell barcode and UMI read is missing, and without it the data cannot be assigned to cells. NCBI SRA kept all three reads of every spot (R1, R2 and I1), and `fastq-dump` with `--split-files` writes them to three separate files. Always check that a public 10x dataset includes the barcode read before you plan an analysis around it.
```

## 4. Download a subset of WT_rep2

The full WT_rep2 run is about 11.8 GB of compressed FASTQ (R2 alone is 8.9 GB). To keep this lab manageable, we download only the **first 10 million spots**. The `fetch_data.sh scrna_fastq` target does this, renames the files to the standard 10x pattern (`<sample>_S1_L001_<read>_001.fastq.gz`) and also downloads the 10x v2 barcode whitelist:

```bash
bash $REPO_DIR/data/fetch_data.sh scrna_fastq     # SCRNA_SPOTS=10000000 by default
```

Under the hood, it runs:

```bash
fastq-dump SRR8257102 -X 10000000 --split-files --gzip -O data/scrna/fastq/WT_rep2
#  -X 10000000     stop after the first 10 million spots
#  --split-files   write each read of a spot to its own file (_1, _2, _3)
#  --gzip          compress the output
```

The download takes 20 to 40 minutes. The first 10 million spots come from the start of the sequencing run, so they are **not a random sample** (they come from the first tiles of the flow cell), but every cell barcode is spread across all tiles, so the subset still contains reads from all of the cells.

In [ ]:
bash $REPO_DIR/data/fetch_data.sh scrna_fastq

# link the FASTQs and the whitelist into the project
ln -sf $DATA_DIR/scrna/fastq/WT_rep2/*.fastq.gz data/raw/
ln -sf $DATA_DIR/scrna/whitelist/737K-august-2016.txt references/
ls -lhL data/raw references

## 5. Look at the reads

Before running any QC program, look at the first record of each file. All three files describe the **same spots in the same order**: line 1 of R1, R2 and I1 has the same read name.

In [ ]:
for r in R1 R2 I1; do
  echo "== $r"
  zcat data/raw/WT_rep2_S1_L001_${r}_001.fastq.gz | head -n 4
done

In [ ]:
# read counts and lengths (-a adds quality columns such as Q30 %)
seqkit stats -a -j 4 data/raw/*.fastq.gz | cut -c1-150

You should see:

| File | Length | Contents |
|---|---|---|
| `R1` | **26 bp** | positions 1-16: cell barcode, positions 17-26: UMI |
| `R2` | **115 bp** | cDNA (the authors sequenced R2 longer than the 98 bp that 10x recommends) |
| `I1` | **8 bp** | the i7 sample index; it is the same few sequences in every read of this sample |

### Splitting R1 into barcode and UMI
Each R1 read is simply the 16 nt barcode followed by the 10 nt UMI. The command below cuts the first 100,000 R1 reads into those two parts and counts how often each barcode appears. A handful of barcodes (the real cells) appear hundreds of times, while a long tail appears only once or twice (empty droplets and sequencing errors).

In [ ]:
zcat data/raw/WT_rep2_S1_L001_R1_001.fastq.gz | head -n 400000 | awk 'NR % 4 == 2 {print substr($0, 1, 16), substr($0, 17, 10)}' | head -n 5

# the 10 most frequent barcodes in the first 100,000 reads
zcat data/raw/WT_rep2_S1_L001_R1_001.fastq.gz | head -n 400000 | awk 'NR % 4 == 2 {print substr($0, 1, 16)}' \
  | sort | uniq -c | sort -k1,1nr | head -n 10

# how many of the first 100,000 barcodes are in the 10x v2 whitelist?
zcat data/raw/WT_rep2_S1_L001_R1_001.fastq.gz | head -n 400000 | awk 'NR % 4 == 2 {print substr($0, 1, 16)}' \
  | awk 'NR == FNR {wl[$1] = 1; next} {n++; if ($1 in wl) ok++} END {printf "%d of %d (%.1f%%) barcodes match the whitelist exactly\n", ok, n, 100 * ok / n}' references/737K-august-2016.txt -

## 6. Quality control with FastQC and MultiQC

We run FastQC on R1 and R2 (the index read is not informative) and combine the reports with MultiQC.

In [ ]:
fastqc -t 4 --outdir results/qc data/raw/WT_rep2_S1_L001_R1_001.fastq.gz data/raw/WT_rep2_S1_L001_R2_001.fastq.gz
multiqc -f -o results/qc results/qc
ls results/qc

Open `results/qc/multiqc_report.html` (and the two FastQC reports) in the Hellbender desktop or the Jupyter file browser. Several modules that would be a concern in bulk RNA-Seq are **expected** for 10x data:

| FastQC module | R1 (barcode + UMI) | R2 (cDNA) |
|---|---|---|
| Per base sequence quality | Should be high (most bases ≥ Q30). Low quality here means barcode and UMI errors | Usually high, dropping slightly towards the 3' end |
| Per base sequence content | Often flagged. The barcodes are drawn from a fixed whitelist, so base frequencies are not perfectly even | Often flagged at the start of the read |
| Per sequence GC content | Not meaningful (barcodes are not biological sequence) | Should be a single peak. Ours is about 40% GC, slightly below Arabidopsis coding sequence (about 44%), because 3' reads cover a lot of AT-rich 3' UTR |
| Sequence duplication levels | Very high: every read from the same cell shares the barcode | High: many reads come from the same molecules (PCR copies) and from highly expressed genes in a small number of cells. UMIs are used to collapse them later |
| Overrepresented sequences / adapter content | Rarely flagged | May show **poly(A)** stretches and the **template switching oligo** (TSO), because some cDNA fragments are short and reach the end of the molecule |

So for 10x data, look mainly at **base quality** (especially of R1) and the number of reads. Duplication and composition warnings are not reasons to discard the data. In our subset, 97.7% of R1 bases and 81.4% of R2 bases were Q30 or better (from `seqkit stats -a` in section 5). A lower value for the 115 bp R2 is normal on the HiSeq 4000, and STAR soft-clips the poor ends.

## 7. Should we trim?

In M2P3 and M3P1 we trimmed bulk RNA-Seq reads with Trimmomatic and fastp. For 10x data, the standard pipelines (Cell Ranger and STARsolo) do **not** trim before alignment:

- **R1 must never be trimmed.** The barcode and UMI are identified by **position** (bases 1-16 and 17-26). Removing even one base, or dropping a read because of a low-quality base, breaks the link between R1 and R2.
- **R2 is aligned with soft-clipping.** STAR can leave unaligned bases at the ends of a read out of the alignment ("soft-clipping"). This removes most of the effect of low-quality ends without a separate trimming step.
- **Adapter, poly(A) and TSO sequences in R2** can be clipped by STARsolo itself with `--clipAdapterType CellRanger4`. This clips the TSO from the 5' end and poly(A) from the 3' end, as Cell Ranger 4 and later do.
- If you do trim R2 with another tool, it must run in **paired mode** with both files, so that any read dropped from R2 is also dropped from R1. Otherwise the files go out of sync and every read is assigned to the wrong barcode.

In this lab we do not trim, and we let STARsolo clip the reads.

## 8. Build the STAR index

We use the same Ensembl Plants release 62 TAIR10 genome as in M5L3 (`fetch_data.sh reference`).

**Which genes to count?** The Ensembl GTF includes protein-coding genes, lncRNAs, tRNAs, rRNAs and other small RNAs. 10x Genomics recommends filtering the GTF to protein-coding (and a few other) gene types before building a Cell Ranger reference (`cellranger mkgtf`), so that reads overlapping both a protein-coding gene and a small RNA gene are not discarded as ambiguous. We do the same with `awk`.

Ryu *et al.* used the original TAIR10 annotation, which has **27,416** protein-coding genes (the length of their `genes.tsv`). Ensembl Plants now distributes the newer **Araport11** annotation on the same TAIR10 genome, with 27,628 protein-coding genes, so our gene list is very similar but not identical.

STAR needs the **read length minus 1** for `--sjdbOverhang`. Our R2 is 115 bp, so we use **114**. (In M5L3 the reads were 100 bp and we used 99.)

In [ ]:
REF=$DATA_DIR/reference/ensembl_plants_62
bash $REPO_DIR/data/fetch_data.sh reference

# unzipped genome and a protein-coding-only GTF
zcat $REF/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz > references/TAIR10.dna.toplevel.fa
zcat $REF/Arabidopsis_thaliana.TAIR10.62.gtf.gz \
  | awk -F'\t' '/^#/ || $9 ~ /gene_biotype "protein_coding"/' > references/TAIR10.62.protein_coding.gtf

# number of genes kept
awk -F'\t' '$3 == "gene"' references/TAIR10.62.protein_coding.gtf | wc -l

In [ ]:
STAR --runMode genomeGenerate \
     --runThreadN 8 \
     --genomeDir references/star_index_tair10_115bp \
     --genomeFastaFiles references/TAIR10.dna.toplevel.fa \
     --sjdbGTFfile references/TAIR10.62.protein_coding.gtf \
     --sjdbOverhang 114 \
     --genomeSAindexNbases 12 \
     --outFileNamePrefix references/star_index_tair10_115bp/

## 9. Align and count with STARsolo

STARsolo is part of STAR. When `--soloType` is set, STAR aligns the cDNA read and, for each read, also reads the barcode and UMI from the barcode read. It then:
1. **Corrects cell barcodes**: barcodes that are not in the whitelist but differ from a whitelist barcode by one base are corrected (sequencing errors)
2. **Assigns reads to genes**: a read counts for a gene if it overlaps the gene's exons on the correct strand (`Gene`), or anywhere in the gene including introns (`GeneFull`, useful for nuclei)
3. **Collapses UMIs**: reads with the same barcode, UMI and gene are counted once, and UMIs that differ by one base from a more abundant UMI are merged (`1MM_CR`, as in Cell Ranger)
4. **Calls cells**: separates real cells from empty droplets (`EmptyDrops_CR`, the method used by Cell Ranger 3 and later)

| Option | Value | Meaning |
|---|---|---|
| `--readFilesIn` | `R2 R1` | **The cDNA read comes first**, then the barcode read. Getting this order wrong is the most common STARsolo mistake |
| `--readFilesCommand` | `zcat` | The FASTQs are gzipped |
| `--soloType` | `CB_UMI_Simple` | One barcode and one UMI at fixed positions in the barcode read (10x 3' and 5') |
| `--soloCBwhitelist` | `737K-august-2016.txt` | The 10x v2 whitelist (v3 uses `3M-february-2018.txt`) |
| `--soloCBstart`, `--soloCBlen` | `1`, `16` | Barcode = bases 1-16 of R1 |
| `--soloUMIstart`, `--soloUMIlen` | `17`, `10` | UMI = bases 17-26 of R1 (use `12` for v3) |
| `--soloBarcodeReadLength` | `26` | Check that R1 is 26 bp |
| `--soloStrand` | `Forward` | 10x 3' reads are on the same strand as the mRNA |
| `--soloFeatures` | `Gene GeneFull` | Count exonic reads, and also exons + introns |
| `--soloCBmatchWLtype` | `1MM_multi_Nbase_pseudocounts` | Barcode correction as in Cell Ranger 3+ |
| `--soloUMIdedup` | `1MM_CR` | UMI collapsing as in Cell Ranger |
| `--soloUMIfiltering` | `MultiGeneUMI_CR` | Remove UMIs that map to several genes, as in Cell Ranger |
| `--soloCellFilter` | `EmptyDrops_CR` | Cell calling as in Cell Ranger 3+ |
| `--soloMultiMappers` | `Unique EM` | Count only unique reads, and also distribute multi-mapping reads by expectation maximization (see M3L1) in a separate matrix |
| `--clipAdapterType` | `CellRanger4` | Clip TSO (5') and poly(A) (3') from R2 |
| `--outSAMtype` | `BAM SortedByCoordinate` | Sorted BAM, as in M5L3 |
| `--outSAMattributes` | `NH HI nM AS CR UR CB UB GX GN sS sQ sM` | Add the raw (`CR`, `UR`) and corrected (`CB`, `UB`) barcode and UMI, and the gene (`GX`, `GN`), to each alignment |
| `--outTmpDir` | `$TMPDIR/...` | Put STAR's temporary files on the node's local disk. STAR uses FIFO (named pipe) files to read gzipped input, and the Hellbender VAST storage does not support them (error: *could not create FIFO file*). The folder must not exist yet |

These settings follow the "Cell Ranger-like" recommendations in the [STARsolo documentation](https://github.com/alexdobin/STAR/blob/master/docs/STARsolo.md).

In [ ]:
STAR --runThreadN 8 \
     --genomeDir references/star_index_tair10_115bp \
     --readFilesIn data/raw/WT_rep2_S1_L001_R2_001.fastq.gz data/raw/WT_rep2_S1_L001_R1_001.fastq.gz \
     --readFilesCommand zcat \
     --soloType CB_UMI_Simple \
     --soloCBwhitelist references/737K-august-2016.txt \
     --soloCBstart 1 --soloCBlen 16 --soloUMIstart 17 --soloUMIlen 10 \
     --soloBarcodeReadLength 26 \
     --soloStrand Forward \
     --soloFeatures Gene GeneFull \
     --soloCBmatchWLtype 1MM_multi_Nbase_pseudocounts \
     --soloUMIdedup 1MM_CR \
     --soloUMIfiltering MultiGeneUMI_CR \
     --soloCellFilter EmptyDrops_CR \
     --soloMultiMappers Unique EM \
     --clipAdapterType CellRanger4 \
     --outSAMtype BAM SortedByCoordinate \
     --outSAMattributes NH HI nM AS CR UR CB UB GX GN sS sQ sM \
     --outFileNamePrefix results/starsolo/WT_rep2_ \
     --outTmpDir ${TMPDIR:-/tmp}/STARtmp_WT_rep2_$$

### Running it as a SLURM job instead
On a shared cluster, long runs are better submitted as batch jobs (see M6L1). Save the following as `scripts/starsolo_WT_rep2.sh`, then submit it with `sbatch scripts/starsolo_WT_rep2.sh` from the project folder:

```bash
#!/bin/bash
#SBATCH --job-name=starsolo_WT_rep2
#SBATCH --partition=general
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH --time=04:00:00
#SBATCH --output=logs/%x_%j.out

module load mamba
source activate module_8_scRNA

STAR --runThreadN ${SLURM_CPUS_PER_TASK} \
     --genomeDir references/star_index_tair10_115bp \
     ...   # the same options as in the cell above
```
Create the `logs/` folder (`mkdir -p logs`) before you submit, or SLURM will not be able to write the log file.

## 10. Reading the STARsolo output

STARsolo writes the usual STAR files (`Log.final.out`, the BAM) plus a `Solo.out/` folder:

```
results/starsolo/WT_rep2_Solo.out/
├── Barcodes.stats              # how many reads had a valid, corrected or unusable barcode
├── Gene/
│   ├── Features.stats          # how many reads were assigned to genes
│   ├── Summary.csv             # the main summary (like Cell Ranger's web_summary)
│   ├── UMIperCellSorted.txt    # UMIs per barcode, sorted (for a knee plot)
│   ├── raw/                    # every barcode: barcodes.tsv, features.tsv, matrix.mtx (+ UniqueAndMult-EM.mtx)
│   └── filtered/               # only the barcodes called as cells
└── GeneFull/                   # the same, counting exons + introns
```

In [ ]:
cat results/starsolo/WT_rep2_Log.final.out | head -n 35

In [ ]:
column -s, -t results/starsolo/WT_rep2_Solo.out/Gene/Summary.csv

**How to read `Summary.csv`** (the last column is what we obtained for the 10-million-spot subset):

| Line | Meaning | Our run |
|---|---|---|
| `Number of Reads` | Spots processed | 10,000,000 |
| `Reads With Valid Barcodes` | Fraction with a whitelist barcode (after correction) | 0.982 (98.2%) |
| `Sequencing Saturation` | How often a new read is a UMI we have already seen; high values mean more sequencing would add few new molecules | 0.095 (9.5%) |
| `Q30 Bases in CB+UMI` | Quality of the barcode read | 0.977 (97.7%) |
| `Reads Mapped to Genome: Unique` | Uniquely aligned R2 | 0.953 (95.3%) |
| `Reads Mapped to Gene: Unique Gene` | Reads assigned to exactly one gene | 0.908 (90.8%) |
| `Estimated Number of Cells` | Barcodes called as cells | 2,034 |
| `Median UMI per Cell` | Molecules detected in a typical cell | 2,156 |
| `Median Gene per Cell` | Genes detected in a typical cell | 1,120 |

Cell Ranger called **1,473** cells for the full WT_rep2 run. With less than a tenth of the reads, each cell has far fewer UMIs, so the cell-calling step has less information to separate cells from empty droplets, and the medians per cell are much lower than the paper's ~24,000 UMIs and ~5,000 genes per cell.

### The knee plot
`UMIperCellSorted.txt` lists the UMI count of every barcode, from largest to smallest. Plotted on log-log axes (rank against UMIs), real cells form a plateau at the top, and empty droplets form a long, low tail. The sharp drop between them is the "knee". We can look at the numbers at a few ranks directly:

In [ ]:
awk 'NR == 1 || NR == 100 || NR == 500 || NR == 1000 || NR == 1500 || NR == 2000 || NR == 5000 || NR == 10000 || NR == 50000 {printf "rank %6d: %6d UMIs\n", NR, $1}' \
    results/starsolo/WT_rep2_Solo.out/Gene/UMIperCellSorted.txt
wc -l < results/starsolo/WT_rep2_Solo.out/Gene/filtered/barcodes.tsv

### Compare with the authors' cells
If you have downloaded the GEO matrices for M8_P1 (`fetch_data.sh scrna_ryu`), you can check how many of your cells were also called by Cell Ranger for the full run. Cell Ranger 2 adds a `-1` suffix to each barcode (the GEM well), which we remove first.

In [ ]:
bash $REPO_DIR/data/fetch_data.sh scrna_ryu
sed 's/-1$//' $DATA_DIR/scrna/GSE123013/Sample_WT-WERGFP_2/filtered_gene_bc_matrices/TAIR10/barcodes.tsv | sort > results/starsolo/cellranger_WT_rep2_barcodes.txt
sort results/starsolo/WT_rep2_Solo.out/Gene/filtered/barcodes.tsv > results/starsolo/starsolo_WT_rep2_barcodes.txt

echo "Cell Ranger (full run): $(wc -l < results/starsolo/cellranger_WT_rep2_barcodes.txt) cells"
echo "STARsolo (subset):      $(wc -l < results/starsolo/starsolo_WT_rep2_barcodes.txt) cells"
echo "called by both:         $(comm -12 results/starsolo/cellranger_WT_rep2_barcodes.txt results/starsolo/starsolo_WT_rep2_barcodes.txt | wc -l)"

In our run, 1,469 of the 2,034 STARsolo cells were also Cell Ranger cells, so STARsolo recovered almost all of the 1,473 cells the authors reported, plus about 560 barcodes that Cell Ranger did not keep. The matrices in `Solo.out/Gene/filtered/` have the same three-file format as the GEO matrices, so the optional exercise in M8_P1 reads your own STARsolo output into Seurat with `Read10X()`.

## Questions:
Answer the following questions in the text box below:
1. Why does `--readFilesIn` list the R2 file before the R1 file? What would STARsolo report if you swapped them?
2. Which FastQC warnings did you get for R1 and R2? For each one, explain whether it is a real problem for a 10x dataset.
3. Why is a missing barcode read (as in the ENA copy of this run) fatal for a scRNA-seq analysis, while a missing index read (I1) is not?
4. Compare `Reads Mapped to Genome: Unique` with `Reads Mapped to Gene: Unique Gene`. Where do the reads that map to the genome but not to a gene come from? How would the `GeneFull` matrix change this?
5. What does the `Sequencing Saturation` value tell you about whether the full run (105 million spots) was sequenced deeply enough?
6. STARsolo called a different number of cells than Cell Ranger. Give two reasons, one related to the data and one to the software.

## References / Resources
- Ryu KH, Huang L, Kang HM, Schiefelbein J (2019). Single-cell RNA sequencing resolves molecular relationships among individual plant cells. *Plant Physiology* 179:1444-1456. [doi:10.1104/pp.18.01482](https://doi.org/10.1104/pp.18.01482). Data: [GSE123013](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE123013), SRA [SRP171040](https://www.ncbi.nlm.nih.gov/sra?term=SRP171040)
- Kaminow B, Yunusov D, Dobin A (2021). STARsolo: accurate, fast and versatile mapping/quantification of single-cell and single-nucleus RNA-seq data. *bioRxiv*. [doi:10.1101/2021.05.05.442755](https://doi.org/10.1101/2021.05.05.442755)
- Dobin A, Davis CA, Schlesinger F, *et al.* (2013). STAR: ultrafast universal RNA-seq aligner. *Bioinformatics* 29:15-21. [doi:10.1093/bioinformatics/bts635](https://doi.org/10.1093/bioinformatics/bts635)
- [STARsolo documentation](https://github.com/alexdobin/STAR/blob/master/docs/STARsolo.md) (Alexander Dobin), including the Cell Ranger-like settings used here
- Lun ATL, Riesenfeld S, Andrews T, *et al.* (2019). EmptyDrops: distinguishing cells from empty droplets in droplet-based single-cell RNA sequencing data. *Genome Biology* 20:63. [doi:10.1186/s13059-019-1662-y](https://doi.org/10.1186/s13059-019-1662-y)
- 10x Genomics barcode whitelist for Single Cell 3' v2 (`737K-august-2016.txt`), from the 10x Genomics [supernova/tenkit repository](https://github.com/10XGenomics/supernova/tree/master/tenkit/lib/python/tenkit/barcodes)
- HBC training: [Introduction to single-cell RNA-seq](https://hbctraining.github.io/Intro-to-scRNAseq/) (CC BY 4.0)